# Statistical Analysis

Two techniques:
1. **Mann-Whitney U test** — is the Self-Service vs. Staff-Assisted resolution-time gap seen in the EDA statistically significant, or could it be noise?
2. **OLS regression on log(resolution time)** — holding other factors constant, which drivers (channel, service type, district, season, weekend, department workload) actually move resolution time, and by how much?

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

conn = sqlite3.connect("../data/processed/sj311.db")

requests = pd.read_sql(
    """
    SELECT
        f.incident_id, f.resolution_hours, f.is_zero_duration,
        f.is_weekend_submission, f.district_id,
        ds.status_name, dst.service_type_name, dep.department_name,
        dc.channel_name, dt.year, dt.quarter, dt.month
    FROM fact_service_requests f
    JOIN dim_status ds ON f.status_id = ds.status_id
    JOIN dim_service_type dst ON f.service_type_id = dst.service_type_id
    JOIN dim_department dep ON f.department_id = dep.department_id
    JOIN dim_channel dc ON f.channel_id = dc.channel_id
    JOIN dim_date dt ON f.date_created_id = dt.date_id
    """,
    conn,
)

closed = requests[
    (requests["status_name"] == "Closed")
    & (~requests["is_zero_duration"])
    & (requests["resolution_hours"] > 0)
].copy()
print(f"{len(closed):,} closed requests with a positive resolution time")

1,019,309 closed requests with a positive resolution time


## 1. Mann-Whitney U test: Self-Service Digital vs. Staff-Assisted

**H0:** resolution time for Self-Service Digital requests is drawn from the same distribution as Staff-Assisted requests (no systematic difference).
**H1:** the two distributions differ.

**Why Mann-Whitney U instead of a t-test:** a t-test compares means and assumes the two groups' distributions are roughly the same shape, just shifted. The EDA boxplot showed the opposite — Self-Service and Staff-Assisted have very different spread and skew, not just different centers. Mann-Whitney U only requires that the data be ordinal/continuous and compares *rank distributions*, which is valid regardless of shape — the right tool when skew this severe is present.

In [2]:
self_service = closed.loc[closed["channel_name"] == "Self-Service Digital", "resolution_hours"]
staff_assisted = closed.loc[closed["channel_name"] == "Staff-Assisted", "resolution_hours"]

u_stat, p_value = stats.mannwhitneyu(self_service, staff_assisted, alternative="two-sided")

# Rank-biserial correlation as an effect size: how much more often does a
# randomly drawn Self-Service request take longer than a randomly drawn
# Staff-Assisted one, versus the reverse. U is on a 0..n1*n2 scale, which
# has no intuitive meaning on its own -- this rescales it to [-1, 1].
n1, n2 = len(self_service), len(staff_assisted)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print(f"n(Self-Service Digital) = {n1:,}, n(Staff-Assisted) = {n2:,}")
print(f"U statistic = {u_stat:,.0f}")
print(f"p-value = {p_value:.3e}")
print(f"rank-biserial effect size = {rank_biserial:.3f}")
print()
print(f"Median hours -- Self-Service Digital: {self_service.median():.2f}, Staff-Assisted: {staff_assisted.median():.4f}")

n(Self-Service Digital) = 685,323, n(Staff-Assisted) = 333,698
U statistic = 202,499,618,640
p-value = 0.000e+00
rank-biserial effect size = -0.771

Median hours -- Self-Service Digital: 145.17, Staff-Assisted: 0.0031


## 2. What drives resolution time? OLS regression on log(hours)

**Why log(resolution_hours) as the target, not raw hours:** OLS assumes roughly normal, homoscedastic residuals. Raw resolution time is right-skewed by orders of magnitude (hours to weeks); modeling its log compresses that range and makes residuals far better behaved. It has a convenient side effect too: a coefficient on a log-target regression is approximately the *percentage* change in resolution time for a one-unit change in that predictor, which is a natural unit for a recommendation ("weekend submissions take X% longer").

**Predictors:**
- `channel_name` — Self-Service Digital vs. Staff-Assisted (drop `Unknown`)
- `service_type_name` — top 8 volume types individually, everything else pooled as `Other` (avoids a 20-level categorical eating degrees of freedom for types with too few observations to estimate reliably)
- `district_id` — categorical (rows with missing geo are naturally excluded)
- `quarter` — captures the seasonal pattern seen in the EDA
- `is_weekend_submission` — staffing-schedule effect
- `log_department_monthly_volume` — a workload/backlog proxy: how many requests that department received in that same calendar month, log-transformed for the same skew reason as the target. The idea: a department swamped with volume in a given month should show slower resolution *in* that month, independent of which service type or district it is.

In [3]:
model_df = closed[closed["channel_name"] != "Unknown"].dropna(subset=["district_id"]).copy()

top_types = model_df["service_type_name"].value_counts().head(8).index
model_df["service_type_grouped"] = np.where(
    model_df["service_type_name"].isin(top_types), model_df["service_type_name"], "Other"
)

dept_monthly_volume = (
    model_df.groupby(["department_name", "year", "month"])["incident_id"]
    .count()
    .rename("department_monthly_volume")
    .reset_index()
)
model_df = model_df.merge(dept_monthly_volume, on=["department_name", "year", "month"], how="left")
model_df["log_department_monthly_volume"] = np.log(model_df["department_monthly_volume"])
model_df["log_resolution_hours"] = np.log(model_df["resolution_hours"])

formula = (
    "log_resolution_hours ~ C(channel_name) + C(service_type_grouped) + C(district_id) "
    "+ C(quarter) + is_weekend_submission + log_department_monthly_volume"
)
model = smf.ols(formula, data=model_df).fit()
print(model.summary())

                             OLS Regression Results                             
Dep. Variable:     log_resolution_hours   R-squared:                       0.445
Model:                              OLS   Adj. R-squared:                  0.445
Method:                   Least Squares   F-statistic:                 1.571e+04
Date:                  Sun, 12 Jul 2026   Prob (F-statistic):               0.00
Time:                          20:47:27   Log-Likelihood:            -1.1567e+06
No. Observations:                450141   AIC:                         2.313e+06
Df Residuals:                    450117   BIC:                         2.314e+06
Df Model:                            23                                         
Covariance Type:              nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [4]:
# Coefficients on a log target aren't directly interpretable as "hours" --
# exponentiating converts each one into a % change in resolution time
# relative to its reference category, holding everything else fixed.
pct_effect = (np.exp(model.params) - 1) * 100
effects = pd.DataFrame({"coef": model.params, "pct_change_resolution_time": pct_effect.round(1)})
effects = effects.drop("Intercept")

reference_levels = {
    "channel_name": (set(model_df["channel_name"].unique()) - {"Staff-Assisted"}),
    "service_type_grouped": set(model_df["service_type_grouped"].unique())
    - {c.split("[T.")[1][:-1] for c in model.params.index if "service_type_grouped" in c},
    "district_id": set(model_df["district_id"].unique())
    - {c.split("[T.")[1][:-1] for c in model.params.index if "district_id" in c},
}
print("Reference (baseline) category per feature:")
for feature, level in reference_levels.items():
    print(f"  {feature}: {level}")
print()
print(effects.sort_values("pct_change_resolution_time"))

Reference (baseline) category per feature:
  channel_name: {'Self-Service Digital'}
  service_type_grouped: {'Abandoned Vehicle'}
  district_id: {'1'}

                                                   coef  \
C(service_type_grouped)[T.Unknown/Other]      -6.539307   
C(service_type_grouped)[T.Other Issues]       -5.216165   
C(channel_name)[T.Staff-Assisted]             -3.487790   
C(service_type_grouped)[T.Graffiti]           -1.867916   
C(service_type_grouped)[T.Illegal Dumping]    -1.685485   
C(service_type_grouped)[T.Pothole]            -1.118058   
C(district_id)[T.3]                           -1.077344   
C(quarter)[T.4]                               -1.065162   
C(service_type_grouped)[T.Other]              -1.034973   
C(district_id)[T.9]                           -1.031547   
C(quarter)[T.3]                               -0.650582   
C(service_type_grouped)[T.Vehicle Concerns]   -0.639869   
C(district_id)[T.4]                           -0.490789   
C(district_id)[T.5]   

## Interpretation

**Hypothesis test:** p ≈ 0 (far below any conventional threshold) — the Self-Service vs. Staff-Assisted gap is not noise. The rank-biserial effect size of **-0.771** is large: a randomly picked Self-Service request outranks (takes longer than) a randomly picked Staff-Assisted request roughly 88% of the time (0.5 + 0.771/2). This is likely a process artifact, not a channel-superiority story — many Staff-Assisted (Agent desktop / Walk-in) tickets look like they're logged and closed in the same interaction, consistent with a staff member resolving something on the spot before entering it, while Self-Service requests wait for a crew dispatch. **This caveat matters for the recommendation:** it argues for auditing *why* self-service tickets sit open, not necessarily for pushing residents toward calling in.

**Regression (R² = 0.445 — the model explains about 45% of the variance in log resolution time):**

- **Weekend submissions take ~219% longer** (holding channel, service type, district, season, and department load constant) — the single largest actionable, controllable effect in the model. This is a staffing-schedule finding: fewer people triaging weekend submissions until the next business day.
- **Streetlight Outage takes ~1,448% longer than Abandoned Vehicle** (the reference category) even after controlling for district and season — confirms the EDA heatmap finding isn't just a district effect; it's specific to that service type's process (likely dependent on a separate utility/contractor dispatch).
- **District 3 (downtown) resolves ~66% faster than District 1**, and District 9 ~64% faster — despite District 3 having the highest raw per-capita request volume (Phase 4, query 04). Raw volume rankings would have pointed at District 3 as the "worst" district; controlling for what's actually being requested and when flips that conclusion. **District 1 and District 6/10 are the real laggards** once confounders are removed — that's the more defensible target for a reallocation recommendation than a raw-volume ranking would have been.
- `log_department_monthly_volume` has a negative coefficient (-15% per unit), i.e. busier department-months look *faster*, not slower. **Treat this one with caution rather than as a clean "more volume = faster" insight**: without department fixed effects, this coefficient is likely picking up that high-volume departments (e.g., trash/junk pickup at scale) run standardized, fast processes, while low-volume specialty categories (encampments, streetlights) are slow for reasons unrelated to their monthly volume. It's a compositional effect, not evidence that flooding a department with requests speeds it up — worth saying exactly this in an interview if asked to defend the coefficient.